[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/26_lora_interview.ipynb)

# 🟠 Solution: LoRA (Low-Rank Adaptation)（面试版）

## 📋 题目描述

**难度：** Medium

实现线性层的 LoRA（低秩适配）。

LoRA 冻结基础权重并添加可训练的低秩矩阵 A 和 B，以极少的参数实现高效微调。

**签名:** `LoRALinear(in_features, out_features, rank, alpha=1.0)`（nn.Module）

**前向传播:** `forward(x) -> Tensor`
- `x` — 输入张量 (*, in_features)

**返回:** `linear(x) + (x @ A^T @ B^T) * (alpha/rank)`

**约束:**
- 基础 `nn.Linear` 权重必须冻结（requires_grad=False）
- `lora_A`：(rank, in_features)，`lora_B`：(out_features, rank) 初始化为零
- 只有 LoRA 参数应接收梯度

**提示：** 1. `linear.weight/bias` 设 `requires_grad_(False)` 冻结
2. `lora_A`: `(rank, in)`，`lora_B`: `(out, rank)` 初始化为零
3. 前向：`linear(x) + (x @ lora_A.T @ lora_B.T) * (alpha/rank)`


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ INTERVIEW

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        # ---- Step 1: 基础层冻结 ----
        self.linear = nn.Linear(in_features, out_features)
        self.linear.weight.requires_grad_(False)
        self.linear.bias.requires_grad_(False)

        # ---- Step 2: LoRA 矩阵 ----
        # 为什么用两个矩阵而不是一个？
        # 一个 (in, out) 矩阵 = 全秩，参数量 = in × out
        # 两个 (in, rank) × (rank, out) = 低秩，参数量 = rank × (in + out)
        # 低秩分解假设：微调时权重变化 ΔW 是低秩的
        # 即模型只需要在一个低维子空间中调整
        self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
        self.scaling = alpha / rank

    def forward(self, x):
        # ---- Step 3: 手写矩阵乘法链 ----
        # x: (*, in_features)
        # x @ lora_A.T: (*, in) @ (in, rank) → (*, rank)
        #   这是降维：将输入投影到 rank 维的低维空间
        # @ lora_B.T: (*, rank) @ (rank, out) → (*, out)
        #   这是升维：从低维空间映射回输出维度
        # × scaling: 控制 LoRA 更新的幅度
        lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return self.linear(x) + lora_out

# 面试核心追问：
# Q: 为什么 lora_B 初始化为零？
# A: 保证训练开始时 LoRA 输出为零，模型行为与预训练模型完全一致
#    这是 LoRA 论文的关键设计——"从预训练模型出发"
# Q: rank 怎么选？
# A: 通常 4-64，越大的 rank 表达能力越强但参数越多
#    实验表明 rank=8-16 就能在大多数任务上取得好效果
# Q: alpha 的作用？
# A: 控制 LoRA 更新的相对强度。alpha/rank 是缩放系数
#    alpha=rank 时 scaling=1，alpha=2*rank 时 LoRA 影响翻倍


In [ ]:
lora = LoRALinear(in_features=256, out_features=256, rank=8)
x = torch.randn(2, 10, 256)
print('Output:', lora(x).shape)
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
total = sum(p.numel() for p in lora.parameters())
print(f'Trainable: {trainable}/{total} ({100*trainable/total:.1f}%)')


In [ ]:
from torch_judge import check
check('lora')


## 📝 核心思路总结

1. **低秩分解**：ΔW = B × A，参数量从 O(in×out) 降到 O(rank×(in+out))
2. **冻结基础层**：只训练 LoRA 参数，预训练知识不丢失
3. **零初始化**：lora_B 初始化为零，保证训练起始点 = 预训练模型
4. **缩放系数**：alpha/rank 控制 LoRA 更新的强度
